In [ ]:
# --- Topographic map (no outer padding, raster fills plot, single colorbar) ---

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from matplotlib.colors import LightSource

dem_file = "DEM_lisboa.tif"   # your DEM

# --- Load DEM and (if needed) reproject to EPSG:4326 ---
with rasterio.open(dem_file) as src:
    dem = src.read(1).astype(float)
    if src.nodata is not None:
        dem[dem == src.nodata] = np.nan

    if src.crs is not None and getattr(src.crs, "to_epsg", lambda: None)() != 4326:
        dst_crs = "EPSG:4326"
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )
        dem_reproj = np.full((height, width), np.nan, dtype=np.float32)
        reproject(
            source=dem,
            destination=dem_reproj,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
            num_threads=2,
        )
        dem = dem_reproj
        # derive extent from transform
        left = transform.c
        top = transform.f
        right = left + transform.a * width
        bottom = top + transform.e * height
        dem_extent = (left, right, bottom, top)
    else:
        b = src.bounds
        dem_extent = (b.left, b.right, b.bottom, b.top)

# --- Hillshade ---
ls = LightSource(azdeg=315, altdeg=45)
dem_for_shade = np.where(np.isfinite(dem), dem, np.nanmin(dem[np.isfinite(dem)]) - 1)
hill = ls.hillshade(dem_for_shade, vert_exag=1.0)

# --- Figure layout: fill entire canvas; reserve a slim strip for the colorbar ---
fig = plt.figure(figsize=(10, 10))
# [left, bottom, width, height] in figure fraction
ax  = fig.add_axes([0.00, 0.00, 0.95, 1.00])   # main map axes fills the figure
cax = fig.add_axes([0.96, 0.05, 0.02, 0.90])   # dedicated colorbar axes

vmin = np.nanpercentile(dem, 2)
vmax = np.nanpercentile(dem, 98)

# Plot hillshade + elevation; make sure image fills the axes
ax.imshow(hill, cmap="gray", extent=dem_extent, origin="upper", aspect="auto")
elev_img = ax.imshow(
    dem,
    cmap="terrain",
    extent=dem_extent,
    origin="upper",
    alpha=0.5,
    vmin=vmin,
    vmax=vmax,
    aspect="auto",
)

# Optional: overlay study area boundary if available
try:
    study_area_ll.boundary.plot(ax=ax, edgecolor="black", linewidth=1.5)
except Exception:
    pass

# Overlay road network (network_gdf) if available
try:
    ng = network_gdf
    # try to ensure network is in lon/lat (EPSG:4326) for alignment with the DEM
    if getattr(ng, "crs", None) is not None:
        try:
            ng = ng.to_crs("EPSG:4326")
        except Exception:
            pass
    # draw major highways a bit thicker, others thinner (fallback to a single style if columns missing)
    try:
        major = ng[ng["highway"].isin(cf_highway_types)]
        minor = ng[~ng.index.isin(major.index)]
        if len(minor):
            minor.plot(ax=ax, color="#222222", linewidth=0.4, alpha=0.6)
        if len(major):
            major.plot(ax=ax, color="#FF0000", linewidth=1.0, alpha=0.9)
    except Exception:
        # simple fallback
        ng.plot(ax=ax, color="#000000", linewidth=0.6, alpha=0.8)
except Exception:
    pass

# Use full DEM extent (or your bbox if you have one — no padding)
try:
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)
except Exception:
    ax.set_xlim(dem_extent[0], dem_extent[1])
    ax.set_ylim(dem_extent[2], dem_extent[3])

# Remove any inner padding/axes decorations
ax.margins(0)
ax.set_aspect("auto")
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

# Single colorbar
cb = fig.colorbar(elev_img, cax=cax)

plt.show()

# Optional: save with zero extra border
fig.savefig("topographic_map.png", dpi=300, bbox_inches="tight", pad_inches=0)

In [ ]:
# Population density map (with better colorbar alignment)

try:
    bgri_local = bgri.copy()
except NameError:
    bgri_local = gpd.read_file("BGRI_clipped.gpkg")

# Ensure numeric types and avoid division by zero
bgri_local["N_INDIVIDUOS"] = pd.to_numeric(bgri_local["N_INDIVIDUOS"], errors="coerce").fillna(0)
bgri_local["SHAPE_Area"]   = pd.to_numeric(bgri_local["SHAPE_Area"], errors="coerce").replace(0, np.nan)

# Population density: persons per km² (SHAPE_Area is m² → divide by 1e6)
bgri_local["pop_km2"] = (
    bgri_local["N_INDIVIDUOS"] / (bgri_local["SHAPE_Area"] / 1e6)
).replace([np.inf, -np.inf], np.nan).fillna(0)

# Optional: also compute per hectare if needed later
bgri_local["pop_ha"] = (
    bgri_local["N_INDIVIDUOS"] / (bgri_local["SHAPE_Area"] / 1e4)
).replace([np.inf, -np.inf], np.nan).fillna(0)

# Project to Web Mercator for basemap
bgri_3857 = bgri_local.to_crs(epsg=3857)

# Clip color scale to 99th percentile for better visual contrast
vmax = np.nanpercentile(bgri_3857["pop_km2"].replace(0, np.nan), 99)
vmax = float(vmax) if np.isfinite(vmax) and vmax > 0 else bgri_3857["pop_km2"].max()

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
bgri_3857.plot(
    column="pop_km2",
    cmap="viridis",
    linewidth=0.35,
    edgecolor="white",
    ax=ax,
    vmin=0,
    vmax=vmax,
    legend=True,
    legend_kwds=dict(
        fraction=0.046,   # width of colorbar
        pad=0.02,         # gap between map and colorbar
        aspect=30,        # colorbar aspect ratio
        shrink=0.83,      # shrink so colorbar aligns with map vertically
        anchor=(1.0, 0.5) # vertical centering next to map
    )
)

# Add basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

# Clean up
ax.set_axis_off()
plt.tight_layout()

# Save figure
fig.savefig("population_density_brgp.png", dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
from matplotlib.lines import Line2D

# Plot network_gdf with customizable hex colors by highway hierarchy and study area on CartoDB Positron
# Edit `highway_colors` to change colors.

fig, ax = plt.subplots(figsize=(12, 12))

# Prepare plotting copies in Web Mercator for contextily basemap
network_plot = network_gdf.to_crs(epsg=3857).copy()
study_area_plot = study_area_gdf.to_crs(epsg=3857).copy()

# Define colors by highway type (hex). Edit these values to change colors.
highway_colors = {
    "motorway": "#d73027",
    "trunk": "#fc8d59",
    "primary": "#fee08b",
    "secondary": "#d9ef8b",
    "tertiary": "#91cf60",
    "residential": "#1a9850",
    "unclassified": "#66c2a5",
    "living_street": "#3288bd",
    "pedestrian": "#5e4fa2",
    # fallback color name for anything not listed
    "other": "#999999"
}

# Normalize highway values to simple lowercase strings for matching
def normalize_hwy(val):
    if pd.isna(val):
        return "other"
    if isinstance(val, (list, tuple, set)) and len(val):
        v = val[0]
    else:
        v = val
    try:
        s = str(v).lower()
    except Exception:
        return "other"
    return s if s in highway_colors else "other"

network_plot["hwy_norm"] = network_plot["highway"].apply(normalize_hwy)

# Plot each highway class with its color
legend_handles = []
plotted = set()
for h, hexcol in highway_colors.items():
    subset = network_plot[network_plot["hwy_norm"] == h]
    if subset.empty:
        continue
    # linewidth can be adjusted; using 1.0 default
    subset.plot(ax=ax, linewidth=1.0, color=hexcol, zorder=3)
    legend_handles.append(Line2D([0], [0], color=hexcol, lw=3, label=h.capitalize()))
    plotted.add(h)

# Plot study area boundary above basemap but under the network lines
study_area_plot.boundary.plot(ax=ax, edgecolor="#000000", linewidth=1, zorder=4, alpha=0.8)

# Add CartoDB Positron basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=network_plot.crs.to_string(), zorder=0)

# Finalize plot
ax.set_axis_off()


# Place legend on the top-left of the map
legend = ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(0.01, 0.99), frameon=True, fontsize=12)
legend.get_frame().set_alpha(0.9)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.ndimage import gaussian_filter
from matplotlib.colors import Normalize

# Prepare POIs in Web Mercator (EPSG:3857)
pois_for_heat = all_pois.copy()
if pois_for_heat.crs is None or pois_for_heat.crs.to_epsg() != 3857:
    pois_for_heat = pois_for_heat.to_crs(epsg=3857)

# Use centroids so polygons contribute as points
pts = pois_for_heat.geometry.centroid
xs = pts.x.to_numpy()
ys = pts.y.to_numpy()

# Study area bounds in Web Mercator
sa_3857 = study_area_gdf.to_crs(epsg=3857)
xmin, ymin, xmax, ymax = sa_3857.total_bounds

# Add a small padding so the study area isn't flush to the image border
pad_frac = 0.03
dx = xmax - xmin
dy = ymax - ymin
pad_x = dx * pad_frac
pad_y = dy * pad_frac

# Histogram range should include the padding so the image covers the displayed axes
hist_range = [[xmin - pad_x, xmax + pad_x], [ymin - pad_y, ymax + pad_y]]

# Build 2D histogram (density grid)
nbins_x = 600
aspect = (hist_range[1][1] - hist_range[1][0]) / (hist_range[0][1] - hist_range[0][0]) if (hist_range[0][1] - hist_range[0][0]) != 0 else 1.0
nbins_y = max(2, int(nbins_x * aspect))

H, xedges, yedges = np.histogram2d(xs, ys, bins=[nbins_x, nbins_y], range=hist_range)

# Smooth density with a Gaussian filter and apply sqrt for better visual contrast
H_smooth = gaussian_filter(H, sigma=3)
H_smooth = np.sqrt(H_smooth)

extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]

# Plot
fig, ax = plt.subplots(figsize=(10, 10))

# Set limits before adding basemap so contextily places tiles correctly
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])

# Add basemap (CartoDB Positron)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

# Choose a lighter colormap and normalize to avoid very dark rendering (clip at 99th percentile)
vmax = max(np.nanpercentile(H_smooth, 99), 1e-9)
norm = Normalize(vmin=0.0, vmax=vmax)
cmap = "YlOrRd"

# Draw heatmap (H needs to be transposed for correct orientation)
im = ax.imshow(
    H_smooth.T,
    origin="lower",
    extent=extent,
    cmap=cmap,
    norm=norm,
    alpha=0.65,
    zorder=2,
    interpolation="bilinear",
)

# Overlay study area boundary
sa_3857.boundary.plot(ax=ax, edgecolor="black", linewidth=1.2, facecolor="none", zorder=4)

ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Create an interactive folium map showing street_lines and POIs (points, polygons, merged polygons).
# Uses existing objects in the notebook: study_centroid, street_lines_4326, pois_points_4326,
# pois_polygons_4326, merged_pois_polygons. Converts merged_pois_polygons to 4326 if needed.

# Center map on study centroid
center = (study_centroid.y, study_centroid.x)
m = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")

# --- Streets layer (GeoJson) ---
street_fg = folium.FeatureGroup(name="street_lines", show=True)
streets_geojson = street_lines_4326[["street_id", "name", "highway", "geometry"]].to_json()
folium.GeoJson(
    streets_geojson,
    name="street_lines",
    style_function=lambda feat: {
        "color": "#1f78b4",
        "weight": 2,
        "opacity": 0.8,
    },
    tooltip=folium.GeoJsonTooltip(fields=["street_id", "name", "highway"],
                                  aliases=["street_id", "name", "highway"],
                                  localize=True)
).add_to(street_fg)
street_fg.add_to(m)

# --- POI points layer (individual CircleMarker) ---
pois_points_fg = folium.FeatureGroup(name="pois_points", show=False)
# iterate and add small circle markers (use 'name' if available)
for idx, row in pois_points_4326.reset_index().iterrows():
    geom = row.geometry
    if geom is None or geom.is_empty:
        continue
    lat, lon = geom.y, geom.x
    label = row.get("name") if "name" in row.index else None
    popup_html = f"<b>poi</b><br>id: {row.get('element id', idx)}"
    if label:
        popup_html += f"<br><b>name:</b> {label}"
    folium.CircleMarker(
        location=(lat, lon),
        radius=3,
        color="#de2d26",
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=300)
    ).add_to(pois_points_fg)
pois_points_fg.add_to(m)

# --- POI polygons layer (GeoJson) ---
pois_polys_fg = folium.FeatureGroup(name="pois_polygons", show=False)
polys_geojson = pois_polygons_4326[["geometry", "name", "leisure"]].to_json()
folium.GeoJson(
    polys_geojson,
    name="pois_polygons",
    style_function=lambda feat: {
        "fillColor": "#31a354",
        "color": "#238b45",
        "weight": 1,
        "fillOpacity": 0.4,
    },
    tooltip=folium.GeoJsonTooltip(fields=["name", "leisure"],
                                  aliases=["name", "type"],
                                  localize=True)
).add_to(pois_polys_fg)
pois_polys_fg.add_to(m)

# --- Merged polygons (result of merging points into polygons) ---
merged_polys_fg = folium.FeatureGroup(name="merged_pois_polygons", show=False)
# ensure merged_pois_polygons is in EPSG:4326 for folium
if merged_pois_polygons.crs is None or merged_pois_polygons.crs.to_epsg() != 4326:
    merged_4326 = merged_pois_polygons.to_crs(epsg=4326)
else:
    merged_4326 = merged_pois_polygons.copy()

# Build GeoJson for merged polygons but provide a custom popup summarizing contributing POIs
for _, row in merged_4326.reset_index().iterrows():
    geom = row.geometry
    if geom is None or geom.is_empty:
        continue
    # summary of contributing_pois if present
    contrib = row.get("contributing_pois", [])
    n_points = sum(1 for c in contrib if c.get("type") == "point") if isinstance(contrib, list) else 0
    n_polys  = sum(1 for c in contrib if c.get("type") == "polygon") if isinstance(contrib, list) else 0
    names = []
    if isinstance(contrib, list):
        for c in contrib:
            attrs = c.get("attributes") or {}
            nm = attrs.get("name") or attrs.get("shop") or attrs.get("leisure") or attrs.get("amenity")
            if nm:
                names.append(str(nm))
    sample_names = ", ".join(sorted(set(names))[:6]) if names else "—"
    popup_html = f"<b>Merged polygon</b><br>points: {n_points}, polygons: {n_polys}<br><b>sample names:</b><br>{sample_names}"
    # add polygon with popup
    gj = folium.GeoJson(
        data=gpd.GeoSeries([geom]).to_json(),
        style_function=lambda feat: {
            "fillColor": "#6a51a3",
            "color": "#54278f",
            "weight": 1,
            "fillOpacity": 0.35,
        }
    )
    gj.add_child(folium.Popup(popup_html, max_width=400))
    gj.add_to(merged_polys_fg)

merged_polys_fg.add_to(m)

# Add layer control and display map
folium.LayerControl(collapsed=False).add_to(m)

# save as html
m.save("lisbon_street_pois_map.html")

In [ ]:
# Create a folium map showing street centerlines, buffers and POIs.
# Uses existing notebook objects: study_centroid, street_lines, street_lines_buffered, all_pois, tooltip_fields, merged_pois_polygons

# center map on study centroid (shapely Point: x=lon, y=lat)
m = folium.Map(location=[study_centroid.y, study_centroid.x], zoom_start=12, tiles="CartoDB positron")

# Convert layers to WGS84 for folium
streets_wgs = street_lines.to_crs(epsg=4326).copy()
buffers_wgs = street_lines_buffered[["street_id","geometry"]].to_crs(epsg=4326).copy()
pois_wgs = all_pois.to_crs(epsg=4326).copy()

# Streets layer (single blue color)
streets_fg = folium.FeatureGroup(name="Streets (centerlines)", show=True)
folium.GeoJson(
    streets_wgs,
    name="streets",
    style_function=lambda feat: {
        "color": "blue",
        "weight": 2,
        "opacity": 0.9
    },
    tooltip=folium.GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_fields)
).add_to(streets_fg)
streets_fg.add_to(m)

# Buffers layer (transparent fill)
buffers_fg = folium.FeatureGroup(name="Street buffers", show=False)
folium.GeoJson(
    buffers_wgs,
    name="buffers",
    style_function=lambda feat: {
        "color": "#3186cc",
        "weight": 1,
        "fillColor": "#3186cc",
        "fillOpacity": 0.08,
        "opacity": 0.6
    },
    tooltip=folium.GeoJsonTooltip(fields=["street_id"], aliases=["street_id"])
).add_to(buffers_fg)
buffers_fg.add_to(m)

# Merged POI polygons layer (from merged_pois_polygons)
merged_pois_fg = folium.FeatureGroup(name="Merged POI polygons", show=False)
if "merged_pois_polygons" in globals() and len(merged_pois_polygons):
    # copy and compute a simple, JSON-serializable attribute for tooltip
    merged_pois_wgs = merged_pois_polygons.to_crs(epsg=4326).copy()
    merged_pois_wgs["contrib_count"] = merged_pois_wgs["contributing_pois"].apply(lambda x: len(x) if isinstance(x, (list, tuple)) else 0)
    # Remove the raw 'contributing_pois' column because it contains shapely geometries
    if "contributing_pois" in merged_pois_wgs.columns:
        merged_pois_wgs = merged_pois_wgs.drop(columns=["contributing_pois"])
    folium.GeoJson(
        merged_pois_wgs,
        name="merged_pois",
        style_function=lambda feat: {
            "color": "#ff0000",
            "weight": 1,
            "fillColor": "#ff0000",
            "fillOpacity": 0.28,
            "opacity": 0.6
        },
        tooltip=folium.GeoJsonTooltip(fields=["contrib_count"], aliases=["contributing POIs"])
    ).add_to(merged_pois_fg)
merged_pois_fg.add_to(m)

# POIs layer: small circle markers; color indicates allowed on motorway/trunk
pois_fg = folium.FeatureGroup(name="POIs", show=True)
for _, row in pois_wgs.iterrows():
    geom = row.geometry
    # some POIs are polygons/multipoints; use representative point
    pt = geom.centroid if not geom.geom_type == "Point" else geom
    lat, lon = pt.y, pt.x
    allowed = bool(row.get("is_allowed", False))
    color = "#ff0000" if allowed else "#ff0000"
    popup_html = f"poi_id: {row.get('poi_id', '')}<br>is_allowed: {allowed}"
    folium.CircleMarker(
        location=[lat, lon],
        radius=3,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(popup_html, parse_html=True)
    ).add_to(pois_fg)

pois_fg.add_to(m)

# Add layer control and display
folium.LayerControl(collapsed=False).add_to(m)

# Saving map to HTML
m.save("map_with_buffers.html")